# Prueba de los modelos desplegados en Hugging Face

**Autor:** Jheison Martinez Bolivar
**Maestría:** Inteligencia Artificial
**Fecha:** septiembre 2026

---

Este cuaderno es la contraparte de `despliegue_spam_sentimiento.ipynb`: allí se entrenaron, evaluaron y empaquetaron los dos modelos; aquí se comprueba que el despliegue funciona **desde fuera**, como lo haría cualquier usuario, sin usar nada del entorno de entrenamiento: se descargan los archivos `model.skops` de los dos repositorios públicos del Hub, se cargan con `skops` y se les pide predecir mensajes nuevos. Los mensajes de prueba son inventados; no vienen de los datasets.

Repositorios publicados:
- Spam en SMS: https://huggingface.co/JheisonM/spam-sms-tfidf-logreg
- Sentimiento en tweets: https://huggingface.co/JheisonM/tweet-sentiment-tfidf-logreg


In [1]:
import warnings

warnings.filterwarnings("ignore", module="tqdm")

import pandas as pd
from IPython.display import display

HF_USER = 'JheisonM'
MODEL_REPOS = {
    'spam': f'{HF_USER}/spam-sms-tfidf-logreg',
    'sentiment': f'{HF_USER}/tweet-sentiment-tfidf-logreg',
}

spam_messages = [
    'Congratulations! Your mobile has won a holiday voucher worth £750. Text CLAIM to 81299 before midnight to collect.',
    'Running 10 min late, can you order me a coffee?',
    'Your account has been locked. Verify at http://bit.ly/x9z within 24h',
    'Mom says dinner is at 7, bring the charger you borrowed',
]
tweets = [
    'This game is absolutely amazing, best purchase this year',
    'The servers are down again, I am done with this company',
    'New update drops next Tuesday according to the patch notes',
    'I love my new coffee machine, anyway what do you think about the Xbox?',
]




### 1. Cargar los modelos desde el Hub

`hf_hub_download` baja el archivo a la caché local y devuelve la ruta; `skops.io.load` lo deserializa. Si el repositorio declarara tipos fuera de la lista de confianza de skops habría que pasarlos en `trusted`; en el cuaderno principal se verificó que no hay ninguno.


In [2]:
import skops.io as sio
from huggingface_hub import hf_hub_download

models = {}
for name, repo_id in MODEL_REPOS.items():
    path = hf_hub_download(repo_id=repo_id, filename='model.skops')
    models[name] = sio.load(path)
    print(f'{name}: {repo_id} -> {models[name].__class__.__name__} con pasos {[step for step, _ in models[name].steps]}')

def probability_table(model, texts):
    proba = model.predict_proba(texts)
    table = pd.DataFrame(proba, columns=model.classes_, index=texts).round(3)
    table['etiqueta'] = model.predict(texts)
    return table

print('Spam / ham (modelo cargado desde el Hub):')
display(probability_table(models['spam'], spam_messages))
print('Sentimiento (modelo cargado desde el Hub):')
display(probability_table(models['sentiment'], tweets))


spam: JheisonM/spam-sms-tfidf-logreg -> Pipeline con pasos ['tfidf', 'clf']


sentiment: JheisonM/tweet-sentiment-tfidf-logreg -> Pipeline con pasos ['tfidf', 'clf']
Spam / ham (modelo cargado desde el Hub):


,ham,spam,etiqueta
Congratulations! Your mobile has won a holiday voucher worth £750. Text CLAIM to 81299 before midnight to collect.,0.079,0.921,spam
"Running 10 min late, can you order me a coffee?",0.738,0.262,ham
Your account has been locked. Verify at http://bit.ly/x9z within 24h,0.504,0.496,ham
"Mom says dinner is at 7, bring the charger you borrowed",0.806,0.194,ham


Sentimiento (modelo cargado desde el Hub):


,Irrelevant,Negative,Neutral,Positive,etiqueta
"This game is absolutely amazing, best purchase this year",0.049,0.038,0.011,0.901,Positive
"The servers are down again, I am done with this company",0.013,0.932,0.019,0.036,Negative
New update drops next Tuesday according to the patch notes,0.053,0.551,0.232,0.164,Negative
"I love my new coffee machine, anyway what do you think about the Xbox?",0.200,0.086,0.036,0.679,Positive


### Cierre

Los dos artefactos publicados cargan fuera del entorno de entrenamiento y predicen con probabilidades. Las predicciones son las mismas que se discutieron en el cuaderno principal, incluidos los casos difíciles (el phishing con tono oficial queda al 50 %, el aviso de parche sale Negative en vez de Neutral): este cuaderno no evalúa calidad, evalúa que lo publicado sea lo que se evaluó.

Una nota sobre la interfaz: la carpeta `app/` trae una app de Gradio que consume estos mismos modelos. Estaba pensada para publicarse como Space, pero Hugging Face reserva hoy los Spaces de Gradio a cuentas PRO (`402 Payment Required` al crearlo), así que no forma parte del despliegue; se puede correr en local con `python app/app.py` y baja los modelos del Hub si no encuentra `models/`.
